In [ ]:
"""

Compute Bayesian weights based on Davison et al. 23 reference => dates 1997+

"""

In [ ]:
import xarray as xr
import numpy as np
from tqdm.notebook import tqdm
import seaborn as sns
import multimelt.useful_functions as uf
import os

In [ ]:
sns.set_context('paper')

In [ ]:
%matplotlib qt5

READ IN DATA

In [ ]:
home_path = '/bettik/burgardc/'
plot_path = '/bettik/burgardc/PLOTS/summer_paper_plots/'
outputpath_GL = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/GL_FLUX/'
outputpath_weights = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/ANALYSIS/'
inputpath_raw = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/'
inputpath_data=home_path+'/DATA/SUMMER_PAPER/interim/'


In [ ]:
inputpath_mask = home_path+'/DATA/SUMMER_PAPER/interim/ANTARCTICA_IS_MASKS/BedMachine_4km/'
file_isf_orig = xr.open_dataset(inputpath_mask+'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc')
nonnan_Nisf = file_isf_orig['Nisf'].where(np.isfinite(file_isf_orig['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = file_isf_orig.sel(Nisf=nonnan_Nisf)
rignot_isf = file_isf_nonnan.Nisf.where(np.isfinite(file_isf_nonnan['isf_area_rignot']), drop=True)
file_isf = file_isf_nonnan.sel(Nisf=rignot_isf)

In [ ]:
# make the domain a little smaller to make the computation even more efficient - file isf has already been made smaller at its creation
map_lim = [-3000000,3000000]

In [ ]:
BedMachine_orig = xr.open_dataset(inputpath_data+'BedMachine_v2_aggregated4km_allvars.nc')
file_BedMachine = uf.cut_domain_stereo(BedMachine_orig, map_lim, map_lim)
file_isf_conc = file_BedMachine['isf_conc']

grid_cell_area_file = xr.open_dataset(inputpath_data+'gridarea_ISMIP6_AIS_4000m_grid.nc').sel(x=file_isf.x,y=file_isf.y)
true_grid_cell_area = grid_cell_area_file['cell_area'].drop('lon').drop('lat')
cell_area_weight = true_grid_cell_area/(4000 * 4000)

lon = file_isf.longitude
lat = file_isf.latitude

xx = file_isf.x
yy = file_isf.y
dx = (xx[2] - xx[1]).values
dy = (yy[2] - yy[1]).values
grid_cell_area_const = abs(dx*dy)  
grid_cell_area_weighted = file_isf_conc * grid_cell_area_const * cell_area_weight

isf_stack_mask = uf.create_stacked_mask(file_isf['ISF_mask'], file_isf.Nisf, ['y','x'], 'mask_coord')


In [ ]:
area_list = []
for kisf in tqdm(file_isf.Nisf):
    true_grid_cell_area_kisf = uf.choose_isf(grid_cell_area_weighted ,isf_stack_mask, kisf) 
    area_list.append(true_grid_cell_area_kisf.sum().assign_coords({'Nisf': kisf}))
area_BedMachine = xr.concat(area_list, dim = 'Nisf')/10**6

In [ ]:
plt.figure()
plt.scatter(np.arange(len(area_BedMachine.Nisf)),area_BedMachine, label='area BedMachine (us)')
plt.scatter(np.arange(len(area_BedMachine.Nisf)),greene_file['area_abs'].mean('time'), label='area Greene averaged over time')
plt.ylim(0,1e5)
plt.xticks(ticks=np.arange(len(file_isf['isf_name'])),labels=file_isf['isf_name'].values, rotation=90)
plt.legend()

In [ ]:
sorted_isf_rignot = [11,69,43,28,12,57,
                     70,44,29,13,58,71,45,30,14,
                     59,72,46,
                     31,
                     15,61,73,47,32,16,48,33,17,62,49,34,18,63,74,
                     50,35,19,64,
                     10,
                     36,20,65,51,37,
                     22,38,52,23,66,53,39,24,
                     67,40,54,75,25,41,
                     26,42,55,68,60,27]

REFERENCE

In [ ]:
inputpath_davison = home_path+'/DATA/SUMMER_PAPER/raw/'
davison_file0 = xr.open_dataset(inputpath_davison + 'steadystate_davison23.nc')
davison_file = xr.open_dataset(inputpath_davison + 'varying_conditions_davison23.nc')
greene_file = xr.open_dataset(inputpath_davison + 'area_isf_greene22.nc')

In [ ]:
mass_balance_davison_steady = davison_file0['basal_melt_obs'] - davison_file0['SMB_obs'] + davison_file0['calving_obs'] - davison_file0['discharge_obs']
mass_balance_davison_steady_unc = np.sqrt(davison_file0['basal_melt_unc']**2 + davison_file0['SMB_unc']**2 + davison_file0['calving_obs']**2 + davison_file0['discharge_unc']**2)

In [ ]:
mass_balance_davison_varying = davison_file['melt_abs'] - davison_file['SMB_abs'] + davison_file0['calving_obs'] - davison_file['discharge_abs']
mass_balance_davison_varying_unc = np.sqrt(davison_file['melt_unc']**2 + davison_file['SMB_unc']**2 + davison_file0['calving_unc']**2 + davison_file['discharge_unc']**2)

SMB

In [ ]:
inputpath_atmo_Chris = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/TS_SMB_DATA/out2/'
inputpath_atmo_Nico = '/bettik/burgardc/DATA/SUMMER_PAPER/interim/SMB_EMULATED/'
scenario = 'historical'

melt_atmo_list = []
for mod in ['CESM2','CNRM-CM6-1','IPSL-CM6A-LR','MPI-ESM1-2-HR','UKESM1-0-LL']:
    if os.path.exists(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-1980-2014.nc'):
        melt_atmo = xr.open_dataset(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
        melt_atmo['time'] = melt_atmo['time'].dt.year
        melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))
    else:
        melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
        melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))
        
for mod in ['ACCESS-ESM1-5','ACCESS-CM2','CESM2-WACCM','CNRM-ESM2-1',
            'CanESM5', 'GFDL-CM4','GFDL-ESM4',
           'MRI-ESM2-0']: #'GISS-E2-1-H',
    melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
    melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))
    
melt_atmo_all = xr.concat(melt_atmo_list, dim='model')



In [ ]:
melt_atmo_all_corr = melt_atmo_all / area_BedMachine * greene_file['area_abs']

BASAL MELT

In [ ]:
param_classic_list = ['linear_local',
              'quadratic_local','quadratic_local_locslope',
              'lazero19',
              'boxes_4_pismyes_picopno']

param_NN_list = ['NNsmall']

In [ ]:
## Melt outputpath
Gt_allmod_list = []
box1_allmod_list = []

scenario = 'historical'
for mod in ['ACCESS-ESM1-5','ACCESS-CM2','CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
            'CanESM5','GFDL-CM4','GFDL-ESM4', 'IPSL-CM6A-LR',
           'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']: #removed 'GISS-E2-1-H',
    
    outputpath_melt = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/OCEAN_MELT_RATE_CMIP/'+mod+'/'

    melt1D_list = []
    for mparam in param_classic_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'eval_metrics_1D_'+mparam+'_'+scenario+'_oneFRIS_anoISMIP.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_classic = xr.concat(melt1D_list, dim='param')       
    Gt_classic = melt1D_classic['melt_1D_Gt_per_y'].sel(time=range(1980,2015))

    melt1D_list = []
    for mparam in param_NN_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'evalmetrics_1D_'+mparam+'_'+scenario+'_anoISMIP.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_NN = xr.concat(melt1D_list, dim='param')   
    Gt_NN = melt1D_NN['predicted_melt'].sel(metrics='Gt')

    Gt_all = xr.concat([Gt_classic, Gt_NN], dim='param')
    
    Gt_allmod_list.append(Gt_all.assign_coords({'model': mod}))

Gt_allmod = xr.concat(Gt_allmod_list, dim='model').sel(time=range(1980,2015))
    

In [ ]:
Gt_allmod_corr = Gt_allmod / area_BedMachine * greene_file['area_abs']

In [ ]:
## Melt outputpath
Gt_allmod_list = []
box1_allmod_list = []

scenario = 'historical'
for mod in ['ACCESS-ESM1-5','ACCESS-CM2','CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
            'CanESM5','GFDL-CM4','GFDL-ESM4', 'IPSL-CM6A-LR',
           'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']: #removed  'GISS-E2-1-H',
    
    outputpath_melt = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/OCEAN_MELT_RATE_CMIP/'+mod+'/'

    melt1D_list = []
    for mparam in param_classic_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'eval_metrics_1D_'+mparam+'_'+scenario+'_oneFRIS_notISMIP.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_classic = xr.concat(melt1D_list, dim='param')       
    Gt_classic = melt1D_classic['melt_1D_Gt_per_y'].sel(time=range(1980,2015))

    melt1D_list = []
    for mparam in param_NN_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'evalmetrics_1D_'+mparam+'_'+scenario+'_notISMIP.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_NN = xr.concat(melt1D_list, dim='param')   
    Gt_NN = melt1D_NN['predicted_melt'].sel(metrics='Gt')

    Gt_all = xr.concat([Gt_classic, Gt_NN], dim='param')
    
    Gt_allmod_list.append(Gt_all.assign_coords({'model': mod}))

Gt_allmod_notISMIP = xr.concat(Gt_allmod_list, dim='model').sel(time=range(1980,2015))
    

In [ ]:
## Melt outputpath
Gt_allmod_list = []
box1_allmod_list = []

scenario = 'historical'
for mod in ['ACCESS-ESM1-5','ACCESS-CM2','CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
            'CanESM5','GFDL-CM4','GFDL-ESM4', 'IPSL-CM6A-LR',
           'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']: #removed 'GISS-E2-1-H',
    
    outputpath_melt = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/OCEAN_MELT_RATE_CMIP/'+mod+'/'

    melt1D_list = []
    for mparam in param_classic_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'eval_metrics_1D_'+mparam+'_'+scenario+'_oneFRIS_anoNEMO.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_classic = xr.concat(melt1D_list, dim='param')       
    Gt_classic = melt1D_classic['melt_1D_Gt_per_y'].sel(time=range(1980,2015))

    melt1D_list = []
    for mparam in param_NN_list:
        melt1D_scenario = xr.open_dataset(outputpath_melt + 'evalmetrics_1D_'+mparam+'_'+scenario+'_anoNEMO.nc')
        melt1D_list.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_NN = xr.concat(melt1D_list, dim='param')   
    Gt_NN = melt1D_NN['predicted_melt'].sel(metrics='Gt')

    Gt_all = xr.concat([Gt_classic, Gt_NN], dim='param')
    
    Gt_allmod_list.append(Gt_all.assign_coords({'model': mod}))

Gt_allmod_anoNEMO = xr.concat(Gt_allmod_list, dim='model').sel(time=range(1980,2015))
    

In [ ]:
Gt_allmod_notISMIP_corr = Gt_allmod_notISMIP / area_BedMachine * greene_file['area_abs']
Gt_allmod_anoNEMO_corr = Gt_allmod_anoNEMO / area_BedMachine * greene_file['area_abs']

In [ ]:
# MERGE, CHOOSING THE LOWEST DIFFERENCE FOR EACH ICE SHELF USING BASAL MELT
diff_mod_obs_bmb_anoISMIP = Gt_allmod_corr.sel(time=slice(1997,2014)).mean('time') - davison_file['melt_abs'].sel(time=slice(1997,2014)).mean('time')
diff_mod_obs_bmb_anoNEMO = Gt_allmod_anoNEMO_corr.sel(time=slice(1997,2014)).mean('time') - davison_file['melt_abs'].sel(time=slice(1997,2014)).mean('time')


ano_choice_list = []

for kisf in file_isf.Nisf:
    #mean_weights_anoNEMO = s_j_bmb_anoNEMO.sel(Nisf=kisf).mean()
    #mean_weights_anoISMIP = s_j_bmb_anoISMIP.sel(Nisf=kisf).mean()
    mean_diff_anoNEMO = diff_mod_obs_bmb_anoNEMO.sel(Nisf=kisf).median()
    mean_diff_anoISMIP = diff_mod_obs_bmb_anoISMIP.sel(Nisf=kisf).median()
    
    #if mean_weights_anoNEMO > mean_weights_anoISMIP:
    if mean_diff_anoNEMO < mean_diff_anoISMIP:
        ano_choice_list.append('NEMO')
    else:
        ano_choice_list.append('ISMIP')


ano_choice = xr.DataArray(data=np.array(ano_choice_list), dims=['Nisf']).assign_coords({'Nisf': file_isf.Nisf})

In [ ]:
ano_choice.rename('ano_choice').to_netcdf(outputpath_weights + 'ano_choice_NEMOorISMIP_withoutGISS.nc')

In [ ]:
for kisf in sorted_isf_rignot:
    print(file_isf['isf_name'].sel(Nisf=kisf).values, ano_choice['ano_choice'].sel(Nisf=kisf).values)

Check the BMB and the ano choice

GL FLUX FOR ABUMIP

In [ ]:
#GL_flux_ABUMIP = xr.open_dataset(outputpath_GL + 'all_GL_fluxes.nc')
GL_flux = xr.open_dataset(outputpath_GL + 'all_GL_fluxes_varying_m.nc')

In [ ]:
GL_flux_ag = xr.Dataset()
GL_flux_ag['flux_Gt_ref'] = xr.concat([GL_flux['flux_Gt_ref_m1'].assign_coords({'m': 1}),
                                       GL_flux['flux_Gt_ref_m3'].assign_coords({'m': 3}),
                                       GL_flux['flux_Gt_ref_m5'].assign_coords({'m': 5})], dim='m')
GL_flux_ag['flux_Gt_ABUMIP'] = xr.concat([GL_flux['flux_Gt_ABUMIP_m1'].assign_coords({'m': 1}),
                                       GL_flux['flux_Gt_ABUMIP_m3'].assign_coords({'m': 3}),
                                       GL_flux['flux_Gt_ABUMIP_m5'].assign_coords({'m': 5})], dim='m')

SUM OF ALL TERMS

In [ ]:
mass_balance_simu_anoISMIP = Gt_allmod_corr - melt_atmo_all_corr + davison_file0['calving_obs'] - GL_flux_ag['flux_Gt_ref']
mass_balance_simu_notISMIP = Gt_allmod_notISMIP_corr - melt_atmo_all_corr + davison_file0['calving_obs'] - GL_flux_ag['flux_Gt_ref']
mass_balance_simu_anoNEMO = Gt_allmod_anoNEMO_corr - melt_atmo_all_corr + davison_file0['calving_obs'] - GL_flux_ag['flux_Gt_ref']

In [ ]:
mass_balance_simu_combined_list = []

for kisf in file_isf.Nisf:
    if ano_choice.sel(Nisf=kisf).values == 'NEMO':
        mass_balance_simu_combined_list.append(mass_balance_simu_anoNEMO.sel(Nisf=kisf))
        
    elif ano_choice.sel(Nisf=kisf).values == 'ISMIP':
        mass_balance_simu_combined_list.append(mass_balance_simu_anoISMIP.sel(Nisf=kisf))

mass_balance_simu_combined = xr.concat(mass_balance_simu_combined_list, dim='Nisf')

In [ ]:
diff_mod_obs = mass_balance_simu_combined.mean('time') - mass_balance_davison_varying.mean('time') 

sigma_obs = mass_balance_davison_varying_unc.mean('time')
sigma_mod = mass_balance_simu_combined.std('time').mean(['model','param','m']) #* mult_fac

In [ ]:
s_j = np.exp(-((diff_mod_obs)**2/(sigma_obs**2 + sigma_mod**2)))

weight = (s_j / (s_j.sum(['model','param','m'])))


calv_below_ABUMIP = davison_file0['calving_obs'] < GL_flux_ag['flux_Gt_ABUMIP']
weight_clean = weight.where(calv_below_ABUMIP,0)
weight_clean.to_dataset(name='bay_weights').to_netcdf(outputpath_weights + 'bayesian_weights_davison_varying_combined_withoutGISS.nc')

In [ ]:
models_til_2300 = ['ACCESS-CM2','ACCESS-ESM1-5','CanESM5','CESM2-WACCM','IPSL-CM6A-LR','MRI-ESM2-0','UKESM1-0-LL'] #'GISS-E2-1-H', 
s_j = np.exp(-((diff_mod_obs)**2/(sigma_obs**2 + sigma_mod**2))).sel(model=models_til_2300)

weight = (s_j / (s_j.sum(['model','param','m'])))

weight_clean = weight.where(calv_below_ABUMIP,0)
weight_clean.to_dataset(name='bay_weights').to_netcdf(outputpath_weights + 'bayesian_weights_davison_varying_combined_2300_withoutGISS.nc')